In [1]:
# conda env 'at3d'

import sys
import os
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import at3d
import MISR_preprocess2 as pp
import util

In [2]:
# INPUT
# MISR NetCDF File (from MISR_preprocess1)
misr_data_file = '../intermediate_data/MISR_OBS_CAMP2Ex_RF17_default.nc'

# Observation wavelength
wavelength = 0.67

# Cloud motion wind vector
s = 1.5 # m/s
theta = 34.8 # degrees
u = s * np.sin(np.radians(theta)) # m/s
v = s * np.cos(np.radians(theta)) # m/s

# OUTPUT
rte_grid_outfile = '../intermediate_data/RTG_1400.nc'
obs_outfile = '../intermediate_data/AT3D_OBS_CAMP2Ex_RF17_default_1400.nc'

In [3]:
print(u)
print(v)

0.8560703515266475
1.231723813700556


In [4]:
# Create radiative transfer grid (for the bounding box)
if(os.path.exists(rte_grid_outfile)):
    print('Grid exists already')
    rte_grid = xr.load_dataset(rte_grid_outfile)
else:
    z = np.array([0, 0.1, 0.2, 0.3, 0.39, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4])
    rte_grid = at3d.grid.make_grid(0.275, 65, 0.275, 65, z)
    rte_grid.to_netcdf(rte_grid_outfile)

# Read in MISR data in Local North coordinates
misr_data_LN = xr.load_dataset(misr_data_file)

# Convert MISR data to ENU coordinates
misr_data_ENU = pp.LN2ENU(misr_data_LN)

# Horizontal shift to place origin at the corner of the rte grid (maintain centering)
misr_data_ENU['E'] = misr_data_ENU.E + np.max(rte_grid.x.data)/2
misr_data_ENU['E'] = misr_data_ENU.E.assign_attrs(units='km')
misr_data_ENU['N'] = misr_data_ENU.N + np.max(rte_grid.y.data)/2
misr_data_ENU['N'] = misr_data_ENU.N.assign_attrs(units='km')

# Project rays to the top of the domain
U2 = np.ones(misr_data_ENU.mu.data.shape)*np.max(rte_grid.z.data)
misr_data_ENU = pp.project_to_U2(misr_data_ENU, U2)

# Perform wind correction
misr_data_ENU = misr_data_ENU.assign_attrs(u=u, v=v)
misr_data_ENU['dt'] = (['camera'], -1*(np.array([0, 60, 113, 159, 204, 249, 295, 348, 408])-204))
misr_data_ENU['dt'] = misr_data_ENU.dt.assign_attrs(units='s')
misr_data_ENU['dt'] = misr_data_ENU.dt.assign_attrs(description='Time shift needed to register at AN camera')
misr_data_ENU['dE'] = misr_data_ENU.dt * u * 1e-3
misr_data_ENU['dE'] = misr_data_ENU.dE.assign_attrs(units='km')
misr_data_ENU['dN'] = misr_data_ENU.dt * v * 1e-3
misr_data_ENU['dN'] = misr_data_ENU.dN.assign_attrs(units='km')
misr_data_ENU['E2'] = misr_data_ENU.E2 + misr_data_ENU.dE
misr_data_ENU['N2'] = misr_data_ENU.N2 + misr_data_ENU.dN

# Calculate the maximum extent at oblique angles and pad
pad = 2 # km
min_extent, max_extent = pp.TOA_extent(rte_grid, misr_data_ENU)
min_extent = min_extent - pad
max_extent = max_extent + pad

# Store observations in a sensor dict
measurements = at3d.containers.SensorsDict()
for cam in misr_data_ENU.camera.data:
    measurements.add_sensor('MISR', pp.make_sensor(misr_data_ENU, cam, min_extent, max_extent, wavelength))

# TODO: add sensor uncertainty model

# Save measurements
# util.save_sensor_dict(obs_outfile, measurements)

Grid exists already


In [5]:
min_extent

array([-2.84721274, -5.78567674, -0.6       ])

In [6]:
max_extent

array([20.8255296 , 23.38180625,  3.4       ])

In [7]:
misr_data_ENU

<xarray.Dataset>
Dimensions:     (camera: 9, y275: 192, x275: 192)
Coordinates:
  * camera      (camera) object 'DF' 'CF' 'BF' 'AF' 'AN' 'AA' 'BA' 'CA' 'DA'
Dimensions without coordinates: y275, x275
Data variables: (12/16)
    brf         (camera, y275, x275) float64 0.1213 0.1352 ... 0.06461 0.06443
    SZA_center  float64 28.64
    SAA_center  float64 129.5
    center_lat  float64 19.27
    center_lon  float64 119.7
    E           (y275, x275) float64 -15.16 -14.88 -14.61 ... 32.21 32.48 32.76
    ...          ...
    E2          (camera, y275, x275) float64 -13.66 -13.39 -13.12 ... 31.4 31.67
    N2          (camera, y275, x275) float64 41.5 41.48 41.45 ... -23.98 -24.01
    U2          (camera, y275, x275) float64 1.4 1.4 1.4 1.4 ... 1.4 1.4 1.4 1.4
    dt          (camera) int64 204 144 91 45 0 -45 -91 -144 -204
    dE          (camera) float64 0.1746 0.1233 0.0779 ... -0.1233 -0.1746
    dN          (camera) float64 0.2513 0.1774 0.1121 ... -0.1774 -0.2513
Attributes:
    orbit:    105254
    path:     117
    blocks:   [75 75]
    u:        0.8560703515266475
    v:        1.231723813700556